<a href="https://colab.research.google.com/github/Saqo12/ltx-video-api/blob/main/ltx_video_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 LTX-Video API Server
Runs on Google Colab (FREE GPU)

**Steps:**
1. Run cell below (install)
2. Run cell 2 (download model)
3. Run cell 3 (ngrok token)
4. Run cell 4 (start server)
5. Copy the `ngrok URL` → use in your TikTok pipeline

In [ ]:
!pip install ngrok pyngrok transformers accelerate pillow requests fastapi uvicorn torch --quiet 2>&1 | tail -3

In [ ]:
import os
os.makedirs('/root/.cache/huggingface', exist_ok=True)

print('Model will download on first request...')
print('Go to Cell 4 → enter your ngrok token → start server')

In [ ]:
# 🔑 Get free ngrok token from https://dashboard.ngrok.com
# Then paste it below:

NGROK_TOKEN = input('Paste your ngrok authtoken: ').strip()

# Configure ngrok
!ngrok config add-authtoken {NGROK_TOKEN}

In [ ]:
import os
import sys
import threading
import subprocess
from google.colab import userdata

# Create app directory
!mkdir -p /content/ltx_api

# Write API app
api_code = '''
import os, sys, io, base64, logging
from pathlib import Path
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import torch
from PIL import Image
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="LTX-Video API")

model = None
pipe = None

class GenRequest(BaseModel):
    prompt: str
    negative_prompt: str = ""
    duration: int = 5
    width: int = 512
    height: int = 896
    num_inference_steps: int = 25
    guidance_scale: float = 7.5

@app.get("/")
def home():
    return {"status": "ok", "service": "LTX-Video API"}

@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/generate")
def generate(req: GenRequest):
    global model, pipe
    if pipe is None:
        try:
            from diffusers import LTXVideoPipeline
            from huggingface_hub import hf_hub_download
            logger.info("Loading LTX-Video model...")
            model_path = hf_hub_download(
                repo_id="Lightricks/LTX-Video",
                filename="ltx-video-2b-v0.9.safetensors",
                cache_dir="/root/.cache/huggingface"
            )
            pipe = LTXVideoPipeline.from_single_file(
                model_path,
                torch_dtype=torch.bfloat16,
                device_map="auto"
            )
            pipe.to("cuda")
            model = True
            logger.info("Model loaded successfully!")
        except Exception as e:
            logger.error(f"Failed to load model: {e}")
            raise HTTPException(status_code=500, detail=f"Model load failed: {str(e)}")

    try:
        num_frames = req.duration * 24
        logger.info(f"Generating {num_frames} frames for: {req.prompt[:50]}...")
        result = pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            num_inference_steps=req.num_inference_steps,
            guidance_scale=req.guidance_scale,
            height=req.height,
            width=req.width,
            num_frames=num_frames
        ).frames[0]

        # Save to bytes
        output = io.BytesIO()
        from PIL import Image
        import numpy as np
        
        # Convert frames to video
        frames = []
        for i, frame in enumerate(result):
            img = Image.fromarray((frame * 255).astype(np.uint8))
            frames.append(img)
        
        # Save as GIF first (simple)
        frames[0].save(
            output,
            format='GIF',
            save_all=True,
            append_images=frames[1:],
            duration=1000/24,
            loop=0
        )
        output.seek(0)
        logger.info("Generation complete!")
        return StreamingResponse(output, media_type="image/gif")

    except Exception as e:
        logger.error(f"Generation error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('/content/ltx_api/main.py', 'w') as f:
    f.write(api_code)

# Start ngrok tunnel in background
import subprocess
ngrok_process = subprocess.Popen(
    ['ngrok', 'http', '8000', '--log', '/tmp/ngrok.log'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

import time
time.sleep(4)

# Get ngrok URL
import requests
try:
    tunnels = requests.get('http://localhost:4040/api/tunnels').json()
    public_url = tunnels['tunnels'][0]['public_url']
    print(f'\n🎉 LTX-Video API LIVE at: {public_url}')
    print(f'\n📌 Use this URL in your pipeline:')
    print(f'   TIKTOK_GENERATOR_API={public_url}')
    print(f'\nEndpoints:')
    print(f'   GET  {public_url}/health     → check if model loaded')
    print(f'   POST {public_url}/generate    → generate video')
    print(f'\nFirst generation takes ~3 mins to download model...')
except Exception as e:
    print(f'Ngrok error: {e}')
    print('Check your ngrok token at https://dashboard.ngrok.com')

# Run server
!cd /content/ltx_api && python main.py &